# Введение и постановка задачи

Сервисы доставки еды уже давно перестали быть просто курьерами, которые привозят заказ. Индустрия e-grocery стремительно идет к аккумулированию и использованию больших данных, чтобы знать о своих пользователях больше и предоставлять более качественные и персонализированные услуги. Одним из шагов к такой персонализации может быть разработка модели, которая понимает привычки и нужды пользователя, и, к примеру, может угадать, что и когда пользователь захочет заказать в следующий раз.

Такая модель, будучи разработанной, может принести значительную ценность для клиента - сэкономить время при сборке корзины, помочь ничего не забыть в заказе, убрать необходимость планировать закупки и следить за заканчивающимися запасами продуктов.

В данном соревновании участникам предлагается решить задачу предсказания следующего заказа пользователя (безотносительно конкретного момента времени, когда этот заказ произойдет). Заказ пользователя состоит из списка уникальных категорий товаров, вне зависимости от того, сколько продуктов каждой категории он взял.


# Описание набора данных

В качестве тренировочных данных представляется датасет с историей заказов 20000 пользователей вплоть до даты отсечки, которая разделяет тренировочные и тестовые данные по времени.

train.csv:

user_id - уникальный id пользователя.

order_completed_at - дата заказа.

cart - список уникальных категорий (category_id), из которых состоял заказ.

В качестве прогноза необходимо для каждой пары пользователь-категория из примера сабмита вернуть 1, если категория будет присутствовать в следующем заказе пользователя, или 0 в ином случае. Список категорий для каждого пользователя примере сабмита - это все категории, которые он когда-либо заказывал.

sample_submission.csv:

Пример сабмита. В тест входят не все пользователи из тренировочных данных, так как некоторые из них так ничего и не заказали после даты отсечки.

id - идентификатор строки - состоит из user_id и category_id, разделенных точкой с запятой: f'{user_id};{category_id}'. Из-за особенностей проверяющей системы Kaggle InClass, использовать колонки user_id, category_id в качестве индекса отдельно невозможно
target - 1 или 0 - будет ли данная категория присутствовать в следующем заказе пользователя


In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
# Блок: Загрузка и первичный анализ train.csv


train = pd.read_csv("train.csv")

print("Структура и информация о train.csv:")
print(train.info())

Структура и информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None


In [ ]:
print(f"Уникальных пользователей: {train['user_id'].nunique()}")

Уникальных пользователей: 20000


In [ ]:
train

,user_id,order_completed_at,cart
0,2,2015-03-22 09:25:46,399
1,2,2015-03-22 09:25:46,14
2,2,2015-03-22 09:25:46,198
3,2,2015-03-22 09:25:46,88
4,2,2015-03-22 09:25:46,157
...,...,...,...
3123059,12702,2020-09-03 23:45:45,441
3123060,12702,2020-09-03 23:45:45,92
3123061,12702,2020-09-03 23:45:45,431
3123062,12702,2020-09-03 23:45:45,24


In [ ]:
# Блок: Дополнительная статистика по train.csv

print("\nДополнительная статистика по train.csv:")

orders_per_user = train['user_id'].value_counts()
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())

print(f"\nУникальных категорий (корзин): {train['cart'].nunique()}")

train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print("\nСтатистика по датам заказов:")
print(f"Период с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")



Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64

Уникальных категорий (корзин): 881

Статистика по датам заказов:
Период с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [ ]:
# Блок: Загрузка и первичный анализ sample_submission.csv

sample_submission = os.path.join(DATA_DIR, "sample_submission.csv")
sub = pd.read_csv(sample_submission)

print("Структура и информация:")
print(sub.info())

Структура и информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [ ]:
sub

,id,target
0,0;133,0
1,0;5,1
2,0;10,0
3,0;396,1
4,0;14,0
...,...,...
790444,19998;26,0
790445,19998;31,0
790446,19998;29,1
790447,19998;798,1


In [ ]:
# Блок: Агрегация корзин и добавление порядкового номера заказа
#
# Назначение:
#   - Сгруппировать данные по пользователям и времени завершения заказа.
#   - Собрать список категорий товаров (корзину) в каждом заказе.
#   - Добавить порядковый номер заказа для каждого пользователя.
#
# Вход:
#   - DataFrame train с данными заказов.
# Выход:
#   - DataFrame orders_agg с агрегированными корзинами и порядковыми номерами.
#
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

orders_agg = (
    train
    .groupby(['user_id', 'order_completed_at'])['cart']
    .apply(list)
    .reset_index()
    .rename(columns={'cart': 'cart_list'})
    .sort_values(['user_id', 'order_completed_at'])
)

orders_agg['order_number'] = (
    orders_agg
    .groupby('user_id')
    .cumcount() + 1
)

user_last_order = (
    orders_agg.groupby('user_id')['order_number'].max()
    .reset_index()
    .rename(columns={'order_number': 'max_order_number'})
)
orders_agg = orders_agg.merge(user_last_order, on='user_id', how='left')

full_orders = orders_agg.copy()

def unique_categories(df):
    return len(set(cat for cl in df['cart_list'] for cat in cl))

print("\nПолная выборка (full):")
print("  - Пользователей:", full_orders['user_id'].nunique())
print("  - Категорий:", unique_categories(full_orders))

print(full_orders.head(10).to_markdown(index=False))



Полная выборка (full):
  - Пользователей: 20000
  - Категорий: 881
|   user_id | order_completed_at   | cart_list                                                                                                         |   order_number |   max_order_number |
|----------:|:---------------------|:------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|
|         0 | 2020-07-19 09:59:17  | [20, 82, 441, 57, 14, 405, 430, 379]                                                                              |              1 |                  3 |
|         0 | 2020-08-24 08:55:32  | [133, 5, 26, 10, 382, 14, 22, 41, 25, 441, 411, 799, 432, 84, 83, 383, 409, 821, 405, 402, 57, 396, 379, 82, 157] |              2 |                  3 |
|         0 | 2020-09-02 07:38:25  | [803, 170, 84, 61, 440, 57, 55, 401, 398, 399, 169]                                                               |              3 

In [ ]:
# Блок: Добавление даты без времени и глобального номера заказа
#
# Назначение:
#   - Преобразовать дату и время заказа к дате без времени.
#   - Создать глобальный порядковый номер заказа по дате для всех пользователей.
#
# Вход:
#   - DataFrame full_orders с историей заказов.
# Выход:
#   - DataFrame full_orders с добавленными колонками order_date и global_order_number.
#
full_orders['order_date'] = full_orders['order_completed_at'].dt.date
full_orders = full_orders.sort_values('order_completed_at')

unique_dates = sorted(full_orders['order_date'].unique())
date_to_global_order_num = {date: idx + 1 for idx, date in enumerate(unique_dates)}
full_orders['global_order_number'] = full_orders['order_date'].map(date_to_global_order_num)


In [ ]:
print(full_orders.head().to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 |
|         3 | 2015-07-04 14:05:22  | [399]                                                                    |              2 |                  7 | 2015-07-04   |                     3 |
|         4 | 2015-07-08 06:59:04  | [54, 55]          

In [ ]:
# Создаем колонку с годом и месяцем для каждого заказа
full_orders['year_month'] = full_orders['order_completed_at'].dt.to_period('M')

In [ ]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number | year_month   |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|:-------------|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 | 2015-03      |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 | 2015-06      |
|         3 | 2015-07-04 14:05:22  | [399]                                                                    |              2 |                  7 | 2015-07-04   |                    

In [ ]:
# --- Индивидуальный порядковый номер месяца для каждого пользователя ---
# Получаем уникальные месяцы для каждого пользователя
unique_user_months = (
    full_orders[['user_id', 'year_month']]
    .drop_duplicates()
    .sort_values(['user_id', 'year_month'])
)

# Добавляем порядковый номер месяца для пользователя
unique_user_months['individual_month'] = unique_user_months.groupby('user_id').cumcount() + 1

# Объединяем обратно с исходным датафреймом
full_orders = full_orders.merge(unique_user_months, on=['user_id', 'year_month'], how='left')

In [ ]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number | year_month   |   individual_month |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|:-------------|-------------------:|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 | 2015-03      |                  1 |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 | 2015-06      |                  1 |
|         3 | 2015-07-04 14:05:22  | [399]                                                          

In [ ]:
# --- Глобальный порядковый номер месяца по всему датафрейму ---
# Получаем уникальные глобальные месяцы по дате (без учета пользователя)
unique_global_months = (
    full_orders[['year_month']]
    .drop_duplicates()
    .sort_values('year_month')
    .reset_index(drop=True)
)
unique_global_months['global_month'] = unique_global_months.index + 1

# Объединяем обратно с основным датафреймом
full_orders = full_orders.merge(unique_global_months, on='year_month', how='left')


In [ ]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number | year_month   |   individual_month |   global_month |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|:-------------|-------------------:|---------------:|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 | 2015-03      |                  1 |              1 |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 | 2015-06      |                  1 |              2 |
|         3 | 2015-07-04 14:05:2

In [ ]:
# Определяем максимальный номер глобального месяца
max_global_month = full_orders['global_month'].max()
print(f"Максимальный номер глобального месяца: {max_global_month}")


# Также посмотрим на последние месяцы
last_months_counts = full_orders[full_orders['global_month'] >= max_global_month - 3].groupby('global_month').agg({
    'user_id': 'nunique',
    'order_completed_at': 'count',
    'order_date': lambda x: x.nunique()  # количество уникальных дней
}).rename(columns={
    'user_id': 'unique_users',
    'order_completed_at': 'total_orders',
    'order_date': 'unique_days'
}).sort_index(ascending=False)

print("\nСтатистика по последним 4 месяцам:")
print(last_months_counts)

# Дополнительная информация по дням в последних месяцах
print("\nКоличество дней активности по месяцам:")
for month in sorted(last_months_counts.index, reverse=True):
    month_data = full_orders[full_orders['global_month'] == month]
    unique_dates = month_data['order_date'].nunique()
    date_range = f"{month_data['order_date'].min()} - {month_data['order_date'].max()}"
    print(f"  Месяц {month}: {unique_dates} дней активности ({date_range})")

Максимальный номер глобального месяца: 60

Статистика по последним 4 месяцам:
              unique_users  total_orders  unique_days
global_month                                         
60                    3169          3403            3
59                   13353         33423           31
58                   14139         33496           31
57                   13392         31155           30

Количество дней активности по месяцам:
  Месяц 60: 3 дней активности (2020-09-01 - 2020-09-03)
  Месяц 59: 31 дней активности (2020-08-01 - 2020-08-31)
  Месяц 58: 31 дней активности (2020-07-01 - 2020-07-31)
  Месяц 57: 30 дней активности (2020-06-01 - 2020-06-30)


In [ ]:
# Проверяем данные за 60-й месяц перед изменением
print("Перед изменением:")
print(f"Количество строк с global_month=60: {len(full_orders[full_orders['global_month'] == 60])}")
print(f"Количество уникальных пользователей в global_month=60: {full_orders[full_orders['global_month'] == 60]['user_id'].nunique()}")
print(f"Диапазон дат в global_month=60: {full_orders[full_orders['global_month'] == 60]['order_date'].min()} - {full_orders[full_orders['global_month'] == 60]['order_date'].max()}")

# Объединяем global_month=60 с global_month=59
full_orders['global_month'] = full_orders['global_month'].apply(lambda x: 59 if x == 60 else x)

# Проверяем после изменения
print("\nПосле изменения:")
print(f"Количество строк с global_month=59: {len(full_orders[full_orders['global_month'] == 59])}")
print(f"Количество уникальных пользователей в global_month=59: {full_orders[full_orders['global_month'] == 59]['user_id'].nunique()}")

# Обновляем max_global_month
max_global_month = full_orders['global_month'].max()
print(f"\nОбновленный максимальный номер глобального месяца: {max_global_month}")

# Проверяем даты в объединенном месяце
month_59_data = full_orders[full_orders['global_month'] == 59]
min_date = month_59_data['order_date'].min()
max_date = month_59_data['order_date'].max()
days_in_month = (pd.to_datetime(max_date) - pd.to_datetime(min_date)).days + 1

print(f"Диапазон дат в global_month=59 после объединения: {min_date} - {max_date}")
print(f"Количество дней в месяце после объединения: {days_in_month} дней")
print(f"Количество уникальных дат: {month_59_data['order_date'].nunique()}")

# Обновляем статистику по месяцам
print("\nОбновленная статистика по месяцам:")
monthly_stats = full_orders[full_orders['global_month'] >= max_global_month - 3].groupby('global_month').agg({
    'user_id': 'nunique',
    'order_completed_at': 'count',
    'order_date': lambda x: x.nunique()
}).rename(columns={
    'user_id': 'unique_users',
    'order_completed_at': 'total_orders',
    'order_date': 'unique_days'
}).sort_index(ascending=False)

print(monthly_stats)

# Также обновляем year_month для строк, которые были перемещены
# Находим year_month для строк с global_month=59 после объединения
# Обновляем year_month на основе дат
month_59_dates = month_59_data['order_completed_at'].dt.to_period('M').unique()
if len(month_59_dates) > 0:
    # Берем наиболее частый year_month в этом месяце
    most_common_month = month_59_data['order_completed_at'].dt.to_period('M').mode()[0]
    print(f"\nНаиболее частый year_month в global_month=59: {most_common_month}")

    # Обновляем year_month для всех записей в global_month=59
    full_orders.loc[full_orders['global_month'] == 59, 'year_month'] = most_common_month

    # Проверяем
    print(f"Уникальные year_month в global_month=59: {full_orders[full_orders['global_month'] == 59]['year_month'].unique()}")

Перед изменением:
Количество строк с global_month=60: 3403
Количество уникальных пользователей в global_month=60: 3169
Диапазон дат в global_month=60: 2020-09-01 - 2020-09-03

После изменения:
Количество строк с global_month=59: 36826
Количество уникальных пользователей в global_month=59: 13799

Обновленный максимальный номер глобального месяца: 59
Диапазон дат в global_month=59 после объединения: 2020-08-01 - 2020-09-03
Количество дней в месяце после объединения: 34 дней
Количество уникальных дат: 34

Обновленная статистика по месяцам:
              unique_users  total_orders  unique_days
global_month                                         
59                   13799         36826           34
58                   14139         33496           31
57                   13392         31155           30
56                    9903         23167           31

Наиболее частый year_month в global_month=59: 2020-08
Уникальные year_month в global_month=59: <PeriodArray>
['2020-08']
Length: 1, 

In [ ]:
# Получаем уникальных пользователей из sub
# В sub в столбце 'id' формат "{user_id};{category_id}"
sub['user_id'] = sub['id'].apply(lambda x: int(x.split(';')[0]))
unique_sub_users = set(sub['user_id'].unique())
print(f"Уникальных пользователей в sub: {len(unique_sub_users)}")

# Получаем уникальных пользователей из предпоследнего global_month=59
users_second_last_month = set(full_orders[full_orders['global_month'] == 59]['user_id'].unique())
print(f"Уникальных пользователей в global_month=59: {len(users_second_last_month)}")

# Находим пересечение пользователей
intersection_users = unique_sub_users.intersection(users_second_last_month)
print(f"Количество пользователей, присутствующих и в sub, и в global_month=59: {len(intersection_users)}")


Уникальных пользователей в sub: 13036
Уникальных пользователей в global_month=59: 13799
Количество пользователей, присутствующих и в sub, и в global_month=59: 10838


In [ ]:
# Проверяем наличие всех пользователей из сабмита в full_orders
print("\n=== Анализ соответствия пользователей между sub и full_orders ===")

# Получаем множество всех пользователей из full_orders
all_full_orders_users = set(full_orders['user_id'].unique())
print(f"Уникальных пользователей в full_orders: {len(all_full_orders_users)}")

# Пользователи из sub, которых нет в full_orders
missing_users = unique_sub_users - all_full_orders_users
print(f"\nПользователи из sub, которых НЕТ в full_orders: {len(missing_users)}")

if len(missing_users) > 0:
    print(f"Примеры отсутствующих пользователей: {sorted(list(missing_users))[:10]}")

    # Проверим, есть ли эти пользователи в исходном train.csv
    missing_in_train = []
    for user_id in sorted(list(missing_users))[:20]:  # проверим первые 20 для примера
        if user_id in train['user_id'].unique():
            missing_in_train.append(user_id)

    if missing_in_train:
        print(f"Некоторые из отсутствующих пользователей есть в train.csv: {missing_in_train[:5]}")

    # Проверим, есть ли заказы у этих пользователей после определенной даты
    if 'missing_in_train' in locals() and missing_in_train:
        print("\nПроверка последних заказов отсутствующих пользователей:")
        for user_id in missing_in_train[:5]:
            user_orders = train[train['user_id'] == user_id]
            if len(user_orders) > 0:
                last_order = user_orders['order_completed_at'].max()
                print(f"  user_id={user_id}: {len(user_orders)} заказов, последний: {last_order}")

# Пользователи, которые есть в full_orders, но нет в sub
extra_users = all_full_orders_users - unique_sub_users
print(f"\nПользователи в full_orders, которых НЕТ в sub: {len(extra_users)}")

# Анализ пересечения
print(f"\n=== Статистика пересечения ===")
print(f"Пользователей только в sub: {len(unique_sub_users - all_full_orders_users)}")
print(f"Пользователей только в full_orders: {len(all_full_orders_users - unique_sub_users)}")
print(f"Общих пользователей: {len(all_full_orders_users.intersection(unique_sub_users))}")

# Более детальный анализ по последним месяцам
print("\n=== Распределение пользователей по месяцам ===")

# Проверяем, в каких месяцах есть пользователи из сабмита
for month in range(max_global_month - 3, max_global_month + 1):
    month_users = set(full_orders[full_orders['global_month'] == month]['user_id'].unique())
    sub_users_in_month = unique_sub_users.intersection(month_users)
    print(f"Месяц {month}: {len(month_users)} пользователей, из них {len(sub_users_in_month)} из sub ({len(sub_users_in_month)/len(month_users)*100:.1f}%)")

# Проверяем активность пользователей из sub в последние месяцы
print("\n=== Активность пользователей из sub в последние месяцы ===")
sub_users_activity = {}

for user_id in list(unique_sub_users)[:50]:  # проверим первые 50 для примера
    user_months = full_orders[full_orders['user_id'] == user_id]['global_month'].unique()
    sub_users_activity[user_id] = len(user_months)

# Анализируем статистику
active_counts = list(sub_users_activity.values())
if active_counts:
    print(f"Среднее количество активных месяцев у пользователей из sub: {np.mean(active_counts):.1f}")
    print(f"Медиана количества активных месяцев: {np.median(active_counts):.1f}")
    print(f"Мин. активных месяцев: {min(active_counts)}")
    print(f"Макс. активных месяцев: {max(active_counts)}")

# Проверяем пользователей из sub, которые вообще не активны в full_orders
inactive_sub_users = []
for user_id in unique_sub_users:
    if user_id not in all_full_orders_users:
        inactive_sub_users.append(user_id)

print(f"\nПользователей из sub, полностью отсутствующих в full_orders: {len(inactive_sub_users)}")
if inactive_sub_users:
    print(f"ID первых 10 неактивных пользователей: {inactive_sub_users[:10]}")

    # Проверим, есть ли у них заказы в train
    inactive_with_orders = []
    for user_id in inactive_sub_users[:20]:
        if user_id in train['user_id'].unique():
            inactive_with_orders.append(user_id)

    if inactive_with_orders:
        print(f"Из них {len(inactive_with_orders)} имеют заказы в train.csv")


=== Анализ соответствия пользователей между sub и full_orders ===
Уникальных пользователей в full_orders: 20000

Пользователи из sub, которых НЕТ в full_orders: 0

Пользователи в full_orders, которых НЕТ в sub: 6964

=== Статистика пересечения ===
Пользователей только в sub: 0
Пользователей только в full_orders: 6964
Общих пользователей: 13036

=== Распределение пользователей по месяцам ===
Месяц 56: 9903 пользователей, из них 6865 из sub (69.3%)
Месяц 57: 13392 пользователей, из них 8795 из sub (65.7%)
Месяц 58: 14139 пользователей, из них 10093 из sub (71.4%)
Месяц 59: 13799 пользователей, из них 10838 из sub (78.5%)

=== Активность пользователей из sub в последние месяцы ===
Среднее количество активных месяцев у пользователей из sub: 13.5
Медиана количества активных месяцев: 8.5
Мин. активных месяцев: 2
Макс. активных месяцев: 46

Пользователей из sub, полностью отсутствующих в full_orders: 0


In [ ]:
# Сгруппируем full_orders по user_id и global_month, объединим корзины (cart_list) за месяц
full_orders_monthly = (
    full_orders
    .groupby(['user_id', 'global_month'])['cart_list']
    .apply(lambda lists: set(cat for sublist in lists for cat in sublist))
    .reset_index()
    .sort_values(['user_id', 'global_month'])
)

In [ ]:
full_orders_monthly

,user_id,global_month,cart_list
0,0,58,"{14, 430, 82, 20, 405, 441, 379, 57}"
1,0,59,"{5, 133, 10, 396, 14, 398, 399, 401, 402, 405,..."
2,1,44,{55}
3,1,52,"{421, 204, 82, 86, 55, 798}"
4,1,53,{55}
...,...,...,...
91125,19995,59,"{0, 67, 712, 393, 398, 14, 84, 57, 31}"
91126,19996,59,"{804, 231, 393, 396, 812, 14, 431, 82, 84, 404..."
91127,19997,59,"{0, 131, 135, 398, 14, 17, 23, 160, 420, 430, ..."
91128,19998,59,"{420, 6, 409, 398, 61, 84, 19, 57, 26, 31, 29,..."


In [ ]:
# Создаем списки для накопления истории и новых колонок
history_cats = set()
history_list = []
overlap_cats_list = []
overlap_cats_counts = []
new_cats_list = []

prev_user_id = None
history_cats = set()

for idx, row in full_orders_monthly.iterrows():
    user = row['user_id']
    current_cats = row['cart_list']

    # Если новый пользователь, сбрасываем историю
    if user != prev_user_id:
        history_cats = set()
        prev_user_id = user

    # Добавляем историю до текущего месяца
    history_list.append(history_cats.copy())

    # Пересечение с историей
    intersection = current_cats.intersection(history_cats)
    overlap_cats_list.append(intersection)
    overlap_cats_counts.append(len(intersection))

    # Новые категории в этом месяце
    new_cats = current_cats.difference(history_cats)
    new_cats_list.append(new_cats)

    # Обновляем историю
    history_cats.update(current_cats)

In [ ]:
# Добавляем новые столбцы с множествами
full_orders_monthly['history_cats'] = history_list
full_orders_monthly['overlap_cats'] = overlap_cats_list
full_orders_monthly['overlap_cats_count'] = overlap_cats_counts
full_orders_monthly['new_cats'] = new_cats_list

In [ ]:
full_orders_monthly

,user_id,global_month,cart_list,history_cats,overlap_cats,overlap_cats_count,new_cats
0,0,58,"{14, 430, 82, 20, 405, 441, 379, 57}",{},{},0,"{14, 430, 82, 20, 405, 441, 379, 57}"
1,0,59,"{5, 133, 10, 396, 14, 398, 399, 401, 402, 405,...","{14, 430, 82, 20, 405, 441, 379, 57}","{14, 82, 405, 441, 379, 57}",6,"{5, 133, 10, 396, 398, 399, 401, 402, 22, 409,..."
2,1,44,{55},{},{},0,{55}
3,1,52,"{421, 204, 82, 86, 55, 798}",{55},{55},1,"{421, 204, 82, 86, 798}"
4,1,53,{55},"{82, 421, 86, 55, 204, 798}",{55},1,{}
...,...,...,...,...,...,...,...
91125,19995,59,"{0, 67, 712, 393, 398, 14, 84, 57, 31}",{},{},0,"{0, 67, 712, 393, 398, 14, 84, 57, 31}"
91126,19996,59,"{804, 231, 393, 396, 812, 14, 431, 82, 84, 404...",{},{},0,"{393, 396, 14, 82, 84, 404, 22, 21, 409, 92, 8..."
91127,19997,59,"{0, 131, 135, 398, 14, 17, 23, 160, 420, 430, ...",{},{},0,"{0, 131, 135, 398, 14, 17, 23, 160, 420, 430, ..."
91128,19998,59,"{420, 6, 409, 398, 61, 84, 19, 57, 26, 31, 29,...",{},{},0,"{420, 6, 409, 398, 61, 84, 19, 57, 26, 31, 29,..."


In [ ]:
max_global_month

59

In [ ]:
# Определяем последние 4 месяца
last_4_months = list(range(max_global_month - 3, max_global_month + 1))
print(f"Последние 4 месяца для анализа: {last_4_months}")

# Выбираем записи только за последние 4 месяца
last_4_months_data = full_orders_monthly[full_orders_monthly['global_month'].isin(last_4_months)].copy()

print(f"\n=== Статистика по последним 4 месяцам ===")
print(f"Всего записей за последние 4 месяца: {len(last_4_months_data)}")
print(f"Уникальных пользователей за последние 4 месяца: {last_4_months_data['user_id'].nunique()}")

Последние 4 месяца для анализа: [56, 57, 58, 59]

=== Статистика по последним 4 месяцам ===
Всего записей за последние 4 месяца: 51233
Уникальных пользователей за последние 4 месяца: 20000


In [ ]:
last_4_months_data

,user_id,global_month,cart_list,history_cats,overlap_cats,overlap_cats_count,new_cats
0,0,58,"{14, 430, 82, 20, 405, 441, 379, 57}",{},{},0,"{14, 430, 82, 20, 405, 441, 379, 57}"
1,0,59,"{5, 133, 10, 396, 14, 398, 399, 401, 402, 405,...","{14, 430, 82, 20, 405, 441, 379, 57}","{14, 82, 405, 441, 379, 57}",6,"{5, 133, 10, 396, 398, 399, 401, 402, 22, 409,..."
7,1,56,"{169, 170, 171, 14, 55, 798}","{421, 231, 169, 204, 812, 14, 82, 19, 23, 86, ...","{169, 798, 14, 55}",4,"{170, 171}"
8,1,58,"{803, 169, 302, 14, 307, 149, 54, 55, 88}","{421, 231, 169, 170, 171, 204, 812, 14, 82, 19...","{88, 169, 14, 55}",4,"{803, 302, 307, 149, 54}"
16,2,57,"{160, 100, 15, 87, 23, 382, 383}","{0, 5, 9, 14, 15, 16, 17, 19, 22, 23, 24, 25, ...","{160, 100, 15, 87, 23, 382, 383}",7,{}
...,...,...,...,...,...,...,...
91125,19995,59,"{0, 67, 712, 393, 398, 14, 84, 57, 31}",{},{},0,"{0, 67, 712, 393, 398, 14, 84, 57, 31}"
91126,19996,59,"{804, 231, 393, 396, 812, 14, 431, 82, 84, 404...",{},{},0,"{393, 396, 14, 82, 84, 404, 22, 21, 409, 92, 8..."
91127,19997,59,"{0, 131, 135, 398, 14, 17, 23, 160, 420, 430, ...",{},{},0,"{0, 131, 135, 398, 14, 17, 23, 160, 420, 430, ..."
91128,19998,59,"{420, 6, 409, 398, 61, 84, 19, 57, 26, 31, 29,...",{},{},0,"{420, 6, 409, 398, 61, 84, 19, 57, 26, 31, 29,..."


In [ ]:
# Находим пользователей, активных во всех последних 4 месяцах
active_users = (
    last_4_months_data
    .groupby('user_id')['global_month']
    .nunique()
    .reset_index()
    .query('global_month == 4')['user_id']
    .tolist()
)

In [ ]:
# Оставляем только таких пользователей и последние 4 месяца
filtered_full_orders_monthly = full_orders_monthly[
    (full_orders_monthly['user_id'].isin(active_users)) &
    (full_orders_monthly['global_month'].isin(last_4_months))
].copy()

In [ ]:
filtered_full_orders_monthly

,user_id,global_month,cart_list,history_cats,overlap_cats,overlap_cats_count,new_cats
98,11,56,"{9, 19, 23, 21, 55}","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{9, 19, 23, 21, 55}",5,{}
99,11,57,"{808, 169, 425, 19, 23, 21, 55}","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{808, 169, 425, 19, 21, 55, 23}",7,{}
100,11,58,"{800, 808, 233, 170, 9, 812, 425, 403, 23, 21,...","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{800, 808, 9, 170, 425, 812, 403, 19, 21, 55, 23}",11,"{233, 90, 244}"
101,11,59,"{169, 810, 171, 9, 403, 21, 55, 23, 410}","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{169, 810, 171, 9, 403, 23, 21, 55}",8,{410}
166,14,56,"{807, 808, 41, 394, 169, 812, 19, 85, 440, 446}","{0, 5, 6, 9, 10, 14, 15, 16, 17, 19, 20, 21, 2...","{807, 808, 41, 394, 169, 812, 19, 85, 440, 446}",10,{}
...,...,...,...,...,...,...,...
82162,15620,59,"{0, 382, 387, 395, 430, 14, 442, 17, 19, 22, 2...","{9, 11, 13, 14, 16, 17, 19, 20, 23, 26, 27, 29...","{395, 430, 14, 17, 19, 23, 409, 442, 61, 382, 57}",11,"{0, 387, 22, 406, 24, 30, 383}"
82166,15622,56,"{0, 385, 5, 9, 11, 395, 23, 25, 29, 157, 425, ...",{},{},0,"{0, 385, 5, 9, 11, 395, 23, 88, 25, 29, 157, 5..."
82167,15622,57,"{0, 5, 9, 10, 12, 13, 14, 15, 16, 19, 20, 21, ...","{0, 385, 5, 9, 11, 395, 23, 88, 25, 29, 157, 5...","{0, 376, 5, 9, 425, 395, 441, 430, 61, 23, 88,...",15,"{10, 12, 13, 14, 15, 16, 271, 19, 20, 21, 22, ..."
82168,15622,58,"{384, 104, 41, 409, 43, 11, 19, 84, 406, 440, ...","{0, 5, 9, 10, 11, 12, 13, 14, 15, 16, 271, 19,...","{41, 11, 43, 19, 84, 799, 440, 409, 382, 57}",10,"{384, 104, 406}"


In [ ]:
print(f"Количество пользователей после фильтрации: {len(active_users)}")
print(f"Размер отфильтрованного датафрейма: {filtered_full_orders_monthly.shape}")

Количество пользователей после фильтрации: 4727
Размер отфильтрованного датафрейма: (18908, 7)


In [ ]:
print("Распределение количества пересечений с историей:")
print(filtered_full_orders_monthly['overlap_cats_count'].describe())

Распределение количества пересечений с историей:
count    18908.000000
mean        21.087582
std         15.187722
min          0.000000
25%         10.000000
50%         19.000000
75%         30.000000
max        113.000000
Name: overlap_cats_count, dtype: float64


In [ ]:
# Группируем по пользователям и считаем std для overlap_cats_count
user_overlap_std = filtered_full_orders_monthly.groupby('user_id')['overlap_cats_count'].std().reset_index()

In [ ]:
user_overlap_std

,user_id,overlap_cats_count
0,11,2.500000
1,14,9.535023
2,16,15.413738
3,21,14.930394
4,35,8.679478
...,...,...
4722,15609,4.123106
4723,15611,4.932883
4724,15615,17.211914
4725,15620,12.922848


In [ ]:
# Определяем порог стабильности — медиану std
std_threshold = user_overlap_std['overlap_cats_count'].median()


In [ ]:
std_threshold

6.582805886043833

In [ ]:
q25 = user_overlap_std['overlap_cats_count'].quantile(0.25)
q75 = user_overlap_std['overlap_cats_count'].quantile(0.75)
mean_val = user_overlap_std['overlap_cats_count'].mean()
std_val = user_overlap_std['overlap_cats_count'].std()

print(f"25-й перцентиль: {q25:.2f}")
print(f"Медиана (50-й перцентиль): {std_threshold:.2f}")
print(f"75-й перцентиль: {q75:.2f}")
print(f"Среднее: {mean_val:.2f}, Стандартное отклонение: {std_val:.2f}")


25-й перцентиль: 4.20
Медиана (50-й перцентиль): 6.58
75-й перцентиль: 9.90
Среднее: 7.50, Стандартное отклонение: 4.54


In [ ]:
# Отбираем пользователей с дисперсией ниже или равной медиане
stable_users = user_overlap_std[user_overlap_std['overlap_cats_count'] <= std_threshold]['user_id']

In [ ]:
print(f"Всего пользователей: {user_overlap_std.shape[0]}")
print(f"Пользователей с низкой дисперсией overlap_cats_count: {len(stable_users)}")

Всего пользователей: 4727
Пользователей с низкой дисперсией overlap_cats_count: 2365


In [ ]:
# Фильтруем исходный датафрейм по стабильным пользователям
stable_behavior_data = filtered_full_orders_monthly[filtered_full_orders_monthly['user_id'].isin(stable_users)].copy()

print(f"Размер датафрейма с пользователями с низкой дисперсией: {stable_behavior_data.shape}")

Размер датафрейма с пользователями с низкой дисперсией: (9460, 7)


In [ ]:
stable_behavior_data

,user_id,global_month,cart_list,history_cats,overlap_cats,overlap_cats_count,new_cats
98,11,56,"{9, 19, 23, 21, 55}","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{9, 19, 23, 21, 55}",5,{}
99,11,57,"{808, 169, 425, 19, 23, 21, 55}","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{808, 169, 425, 19, 21, 55, 23}",7,{}
100,11,58,"{800, 808, 233, 170, 9, 812, 425, 403, 23, 21,...","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{800, 808, 9, 170, 425, 812, 403, 19, 21, 55, 23}",11,"{233, 90, 244}"
101,11,59,"{169, 810, 171, 9, 403, 21, 55, 23, 410}","{9, 10, 14, 271, 16, 272, 17, 19, 21, 23, 25, ...","{169, 810, 171, 9, 403, 23, 21, 55}",8,{410}
360,37,56,"{0, 5, 11, 396, 14, 398, 16, 15, 402, 403, 19,...","{384, 385, 386, 388, 5, 398, 15, 16, 17, 14, 1...","{5, 398, 14, 16, 15, 402, 403, 19, 22, 24, 409...",24,"{0, 67, 420, 11, 396, 431, 405, 85, 86, 440, 2..."
...,...,...,...,...,...,...,...
82130,15609,59,"{64, 420, 808, 425, 42, 441, 812, 715, 432, 21...","{64, 388, 391, 392, 395, 715, 14, 16, 17, 403,...","{64, 420, 808, 425, 715, 432, 57, 61}",8,"{441, 42, 812, 21}"
82133,15611,56,"{9, 14, 17, 86, 409, 61, 31}",{},{},0,"{17, 86, 9, 409, 61, 14, 31}"
82134,15611,57,"{9, 398, 15, 16, 17, 14, 19, 22, 23, 409, 412,...","{17, 86, 9, 409, 61, 14, 31}","{9, 14, 17, 86, 409, 61, 31}",7,"{89, 77, 398, 15, 16, 430, 431, 19, 432, 55, 2..."
82135,15611,58,"{384, 89, 803, 71, 43, 14, 398, 16, 84, 437, 5...","{9, 77, 14, 398, 15, 17, 16, 19, 84, 86, 22, 2...","{398, 14, 16, 84, 89, 61, 57}",7,"{384, 803, 71, 43, 437}"


In [ ]:
print("Распределение количества пересечений с историей:")
print(stable_behavior_data['overlap_cats_count'].describe())

Распределение количества пересечений с историей:
count    9460.000000
mean       17.275053
std        13.148168
min         0.000000
25%         8.000000
50%        14.000000
75%        25.000000
max        89.000000
Name: overlap_cats_count, dtype: float64


In [ ]:
# Функция для вывода истории пользователя (месяц и cart_list)
def print_user_order_history(user_id, df, title):
    user_data = df[df['user_id'] == user_id].sort_values('global_month')
    print(f"\nИстория заказов {title} (user_id={user_id}):")
    for _, row in user_data.iterrows():
        month = row['global_month']
        cart = sorted(row['cart_list'])
        print(f"  Месяц {month}: корзина категорий {cart}")

# Выбираем стабильного и нестабильного пользователя (как в предыдущем коде)
stable_user_id = stable_users.iloc[0]
unstable_user_id = user_overlap_std[user_overlap_std['overlap_cats_count'] > std_threshold]['user_id'].iloc[0]

# Выводим истории
print_user_order_history(stable_user_id, filtered_full_orders_monthly, "стабильного пользователя")
print_user_order_history(unstable_user_id, filtered_full_orders_monthly, "нестабильного пользователя")




История заказов стабильного пользователя (user_id=11):
  Месяц 56: корзина категорий [9, 19, 21, 23, 55]
  Месяц 57: корзина категорий [19, 21, 23, 55, 169, 425, 808]
  Месяц 58: корзина категорий [9, 19, 21, 23, 55, 90, 170, 233, 244, 403, 425, 800, 808, 812]
  Месяц 59: корзина категорий [9, 21, 23, 55, 169, 171, 403, 410, 810]

История заказов нестабильного пользователя (user_id=14):
  Месяц 56: корзина категорий [19, 41, 85, 169, 394, 440, 446, 807, 808, 812]
  Месяц 57: корзина категорий [55, 169, 394, 446, 804]
  Месяц 58: корзина категорий [0, 19, 67, 169, 170, 256, 382, 408, 417, 422, 430]
  Месяц 59: корзина категорий [5, 19, 22, 41, 55, 57, 61, 67, 82, 84, 85, 87, 104, 169, 170, 198, 209, 253, 382, 394, 402, 430, 431, 432, 440, 441, 445, 798, 802, 807]


In [ ]:
# Извлечь все категории из множества cart_list для всех строк и объединить в один набор
unique_categories = set(cat for cats_set in stable_behavior_data['cart_list'] for cat in cats_set)

print(f"Количество уникальных категорий в stable_behavior_data: {len(unique_categories)}")


Количество уникальных категорий в stable_behavior_data: 663


In [ ]:
import pandas as pd

# 1. Подсчет общего количества заказов (вхождений cart_list) для КАЖДОЙ КАТЕГОРИИ в каждом месяце
# Сначала развернем cart_list
exploded_orders = full_orders_monthly.explode('cart_list').reset_index(drop=True)
exploded_orders = exploded_orders.rename(columns={'cart_list': 'category_id'})
exploded_orders['category_id'] = exploded_orders['category_id'].astype(int)

# Подсчитаем количество заказов (строк) для каждой пары (global_month, category_id)
monthly_cat_counts = exploded_orders.groupby(['global_month', 'category_id']).size().reset_index(name='cat_orders_in_month')

# Создадим словарь {(global_month, category_id): cat_orders_in_month} для быстрого доступа
weight_lookup = {(row['global_month'], row['category_id']): row['cat_orders_in_month']
                 for _, row in monthly_cat_counts.iterrows()}

# 2. Функция для создания обучающей выборки из одной строки
def create_training_rows(row):
    user_id = row['user_id']
    month = row['global_month']
    history_cats = row['history_cats'] # Берем history_cats из строки
    overlap_cats = row['overlap_cats']

    rows = []
    # Итерируемся только по категориям из history_cats
    for cat in history_cats:
        target = 1 if cat in overlap_cats else 0
        # Вычисляем вес: global_month * количество_заказов_этой_конкретной_категории_в_этом_месяце
        # Используем .get() с дефолтным значением 0, если пара (month, cat) не найдена
        cat_orders_in_month = weight_lookup.get((month, cat), 0)
        weight = cat_orders_in_month
        rows.append({
            'user_id': user_id,
            'global_month': month,
            'history_cats': history_cats, # Добавляем полное множество истории
            'category': cat,
            'target': target,
            'weight': weight
        })
    return rows

# 3. Применение функции ко всем строкам датафрейма
all_training_rows = []
for _, row in full_orders_monthly.iterrows():
    # Пропускаем строки, где history_cats пустое множество
    if row['history_cats']:
        all_training_rows.extend(create_training_rows(row))

# 4. Создание нового датафрейма
train_df = pd.DataFrame(all_training_rows)

# 5. Вывод информации о полученной выборке
print(f"Размер обучающей выборки: {train_df.shape}")
print(f"Количество строк с меткой 1: {train_df['target'].sum()}")
print(f"Количество строк с меткой 0: {(train_df['target'] == 0).sum()}")
print("\nПример первых строк:")
print(train_df.head(10))

Размер обучающей выборки: (3590426, 6)
Количество строк с меткой 1: 1174558
Количество строк с меткой 0: 2415868

Пример первых строк:
   user_id  global_month                          history_cats  category  \
0        0            59  {14, 430, 82, 20, 405, 441, 379, 57}        14   
1        0            59  {14, 430, 82, 20, 405, 441, 379, 57}       430   
2        0            59  {14, 430, 82, 20, 405, 441, 379, 57}        82   
3        0            59  {14, 430, 82, 20, 405, 441, 379, 57}        20   
4        0            59  {14, 430, 82, 20, 405, 441, 379, 57}       405   
5        0            59  {14, 430, 82, 20, 405, 441, 379, 57}       441   
6        0            59  {14, 430, 82, 20, 405, 441, 379, 57}       379   
7        0            59  {14, 430, 82, 20, 405, 441, 379, 57}        57   
8        1            52                                  {55}        55   
9        1            53           {82, 421, 86, 55, 204, 798}        82   

   target  weight  
0       

In [ ]:
train_df

,user_id,global_month,history_cats,category,target,weight
0,0,59,"{14, 430, 82, 20, 405, 441, 379, 57}",14,1,8392
1,0,59,"{14, 430, 82, 20, 405, 441, 379, 57}",430,0,4864
2,0,59,"{14, 430, 82, 20, 405, 441, 379, 57}",82,1,2986
3,0,59,"{14, 430, 82, 20, 405, 441, 379, 57}",20,0,2333
4,0,59,"{14, 430, 82, 20, 405, 441, 379, 57}",405,1,525
...,...,...,...,...,...,...
3590421,19478,59,"{391, 9, 426, 14, 84, 437, 54, 23, 405, 55, 22...",55,1,4581
3590422,19478,59,"{391, 9, 426, 14, 84, 437, 54, 23, 405, 55, 22...",22,0,6777
3590423,19478,59,"{391, 9, 426, 14, 84, 437, 54, 23, 405, 55, 22...",381,0,504
3590424,19478,59,"{391, 9, 426, 14, 84, 437, 54, 23, 405, 55, 22...",382,0,5703


In [ ]:
print("\nПример структуры датафрейма:")
print(train_df.info())


Пример структуры датафрейма:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3590426 entries, 0 to 3590425
Data columns (total 6 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   user_id       int64 
 1   global_month  int64 
 2   history_cats  object
 3   category      int64 
 4   target        int64 
 5   weight        int64 
dtypes: int64(5), object(1)
memory usage: 164.4+ MB
None


In [ ]:
import pandas as pd


# 2. Проверка: Сравнение конкретного пользователя в месяце 59
check_user_id = 0
check_month = 59

print(f"--- Проверка: Сравнение user_id={check_user_id}, global_month={check_month} ---")

# Найдем строку в full_orders_monthly
user_month_row = full_orders_monthly[
    (full_orders_monthly['user_id'] == check_user_id) &
    (full_orders_monthly['global_month'] == check_month)
]

if not user_month_row.empty:
    print("Строка из full_orders_monthly:")
    print(user_month_row[['user_id', 'global_month', 'history_cats', 'overlap_cats']].to_string(index=False))
    history_cats_orig = user_month_row['history_cats'].iloc[0]
    overlap_cats_orig = user_month_row['overlap_cats'].iloc[0]
    print(f"  - history_cats (из full_orders_monthly): {sorted(history_cats_orig)}")
    print(f"  - overlap_cats (из full_orders_monthly): {sorted(overlap_cats_orig)}")
else:
    print(f"Пользователь {check_user_id} не найден в месяце {check_month} в full_orders_monthly.")
    history_cats_orig = set()
    overlap_cats_orig = set()

# Найдем соответствующие строки в train_df
train_user_month = train_df[
    (train_df['user_id'] == check_user_id) &
    (train_df['global_month'] == check_month)
]

if not train_user_month.empty:
    print("\nСоответствующие строки в train_df:")
    print(train_user_month[['user_id', 'global_month', 'category', 'target', 'weight']].to_string(index=False))

    # Проверим, что все category из train_df присутствуют в history_cats_orig
    categories_in_train = set(train_user_month['category'])
    categories_in_history = history_cats_orig
    print(f"\n  - Все 'category' из train_df присутствуют в 'history_cats' из full_orders_monthly: {categories_in_train.issubset(categories_in_history)}")

    # Проверим, что target=1 соответствует overlap_cats_orig
    target_1_cats = set(train_user_month[train_user_month['target'] == 1]['category'])
    overlap_cats_from_train = target_1_cats
    print(f"  - Все 'category' с 'target=1' из train_df присутствуют в 'overlap_cats' из full_orders_monthly: {overlap_cats_from_train.issubset(overlap_cats_orig)}")
    print(f"  - Все 'category' из 'overlap_cats' из full_orders_monthly имеют 'target=1' в train_df: {overlap_cats_orig.issubset(target_1_cats)}")

else:
    print(f"Не найдено строк в train_df для user_id={check_user_id}, global_month={check_month}.")

print("\n" + "="*50 + "\n")


# 3. Сравнение топ-категорий: target=1 vs. суммарный вес (для 58-го месяца)
month_to_check = 58
print(f"--- Сравнение топ-категорий по target=1 и по суммарному весу для месяца {month_to_check} ---")

# Топ-категорий по target=1
train_month_58 = train_df[train_df['global_month'] == month_to_check]
top_cats_target_1 = train_month_58[train_month_58['target'] == 1]['category'].value_counts().head(20)
print("Топ-20 категорий по target=1 (месяц 58):")
print(top_cats_target_1.index.tolist())

# Топ-категорий по суммарному весу
weight_by_cat = train_month_58.groupby('category')['weight'].sum().reset_index()
weight_by_cat_sorted = weight_by_cat.sort_values(by='weight', ascending=False)
top_cats_by_weight = weight_by_cat_sorted.head(20)
print("\nТоп-20 категорий по суммарному весу (месяц 58):")
print(top_cats_by_weight['category'].tolist())

# Пересечение топов
top_target_1_set = set(top_cats_target_1.index)
top_weight_set = set(top_cats_by_weight['category'])
intersection_top = top_target_1_set.intersection(top_weight_set)
print(f"\nПересечение ID топ-20 (target=1) и топ-20 (по весу) для месяца {month_to_check}: {len(intersection_top)}")
print(f"Общие ID категорий: {sorted(list(intersection_top))}")


--- Проверка: Сравнение user_id=0, global_month=59 ---
Строка из full_orders_monthly:
 user_id  global_month                         history_cats                overlap_cats
       0            59 {14, 430, 82, 20, 405, 441, 379, 57} {14, 82, 405, 441, 379, 57}
  - history_cats (из full_orders_monthly): [14, 20, 57, 82, 379, 405, 430, 441]
  - overlap_cats (из full_orders_monthly): [14, 57, 82, 379, 405, 441]

Соответствующие строки в train_df:
 user_id  global_month  category  target  weight
       0            59        14       1    8392
       0            59       430       0    4864
       0            59        82       1    2986
       0            59        20       0    2333
       0            59       405       1     525
       0            59       441       1    1716
       0            59       379       1    3398
       0            59        57       1    9162

  - Все 'category' из train_df присутствуют в 'history_cats' из full_orders_monthly: True
  - Все 'category' 

In [ ]:
from collections import defaultdict

In [ ]:
# --- 1. Определение уникальных категорий и индексация ---
# Предполагается, что 'train' содержит столбец 'cart' с категориями
unique_categories_in_train = set(train['cart'].unique())
num_unique_categories = len(unique_categories_in_train)

# Создаем отображение категории -> индекс
cat_to_idx = {cat: idx for idx, cat in enumerate(sorted(unique_categories_in_train))} # Сортировка для детерминированности
idx_to_cat = {idx: cat for cat, idx in cat_to_idx.items()} # Обратное отображение

print(f"Количество уникальных категорий в train: {num_unique_categories}")
print(f"Пример отображения (первые 5): {dict(list(cat_to_idx.items())[:5])}")

# --- 2. Подготовка векторов X и y ---
# Предполагается, что 'full_orders_monthly' содержит столбцы 'user_id', 'global_month', 'history_cats', 'overlap_cats'
# И что 'history_cats' и 'overlap_cats' являются множествами (set) объектов Python.

X_vectors = []
y_vectors = []
processed_rows = []

for idx, row in full_orders_monthly.iterrows():
    user_id = row['user_id']
    global_month = row['global_month']
    history_cats_set = row['history_cats']
    overlap_cats_set = row['overlap_cats']

    # Инициализация векторов X и y
    X_vec = np.zeros(num_unique_categories, dtype=int)
    y_vec = np.zeros(num_unique_categories, dtype=int)

    # Заполнение вектора X (история)
    for cat in history_cats_set:
        if cat in cat_to_idx: # Проверяем, что категория из history есть в train
            idx_in_vector = cat_to_idx[cat]
            X_vec[idx_in_vector] = 1

    # Заполнение вектора y (цель - пересечение в следующем месяце)
    for cat in overlap_cats_set:
        if cat in cat_to_idx: # Проверяем, что категория из overlap есть в train
            idx_in_vector = cat_to_idx[cat]
            y_vec[idx_in_vector] = 1

    X_vectors.append(X_vec)
    y_vectors.append(y_vec)
    # Сохраняем также user_id и global_month для контекста
    processed_rows.append({'user_id': user_id, 'global_month': global_month})

print(f"\nОбработано строк: {len(processed_rows)}")

# --- 3. Формирование итогового датасета (опционально) ---
# Можно создать датафрейм с user_id, global_month, X (вектор как объект), y (вектор как объект)
# Или сохранить X_vectors, y_vectors, processed_rows отдельно для использования в модели

processed_df = pd.DataFrame(processed_rows)
processed_df['X_vector'] = X_vectors
processed_df['y_vector'] = y_vectors

print(f"\nРазмер итогового датафрейма: {processed_df.shape}")
print(processed_df[['user_id', 'global_month']].head(2)) # Показываем только user_id и global_month для краткости
# print(processed_df.head(2)) # Не распечатываем векторы целиком, они длинные

Количество уникальных категорий в train: 881
Пример отображения (первые 5): {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4}

Обработано строк: 91130

Размер итогового датафрейма: (91130, 4)
   user_id  global_month
0        0            58
1        0            59


In [ ]:
processed_df

,user_id,global_month,X_vector,y_vector
0,0,58,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,0,59,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
2,1,44,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,1,52,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,1,53,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
91125,19995,59,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
91126,19996,59,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
91127,19997,59,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
91128,19998,59,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
import os

# Убедимся, что директория существует
processed_data_dir = "/content/drive/MyDrive/Colab Notebooks/sm/data/processed"
os.makedirs(processed_data_dir, exist_ok=True)

In [ ]:
# Сохраняем датафрейм
processed_df_path = os.path.join(processed_data_dir, "processed_features_df.csv")
# Для сохранения списков/векторов в CSV часто используют строковое представление.
# Сохраним вектора как строки, их можно будет восстановить с помощью eval() при загрузке.
processed_df_save = processed_df.copy()
processed_df_save['X_vector'] = processed_df_save['X_vector'].apply(lambda x: x.tolist()) # Конвертируем numpy array в list
processed_df_save['y_vector'] = processed_df_save['y_vector'].apply(lambda x: x.tolist())
processed_df_save.to_csv(processed_df_path, index=False)
print(f"Датасет сохранен в: {processed_df_path}")

Датасет сохранен в: /content/drive/MyDrive/Colab Notebooks/sm/data/processed/processed_features_df.csv


In [ ]:
# --- Сохранение словарей индексации ---
cat_to_idx_path = os.path.join(processed_data_dir, "cat_to_idx.json")
idx_to_cat_path = os.path.join(processed_data_dir, "idx_to_cat.json")

import json

# Преобразуем ключи и значения в int при сохранении, если они numpy.int64
cat_to_idx_serializable = {int(k): int(v) for k, v in cat_to_idx.items()}
idx_to_cat_serializable = {int(k): int(v) for k, v in idx_to_cat.items()}

with open(cat_to_idx_path, 'w') as f:
    json.dump(cat_to_idx_serializable, f) # Сохраняем преобразованный словарь
with open(idx_to_cat_path, 'w') as f:
    json.dump(idx_to_cat_serializable, f) # Сохраняем преобразованный словарь

print(f"Словарь cat_to_idx сохранен в: {cat_to_idx_path}")
print(f"Словарь idx_to_cat сохранен в: {idx_to_cat_path}")

# Также сохраним количество уникальных категорий как отдельный файл для удобства
num_cats_path = os.path.join(processed_data_dir, "num_unique_categories.txt")
with open(num_cats_path, 'w') as f:
    f.write(str(num_unique_categories))

print(f"Количество уникальных категорий сохранено в: {num_cats_path}")

Словарь cat_to_idx сохранен в: /content/drive/MyDrive/Colab Notebooks/sm/data/processed/cat_to_idx.json
Словарь idx_to_cat сохранен в: /content/drive/MyDrive/Colab Notebooks/sm/data/processed/idx_to_cat.json
Количество уникальных категорий сохранено в: /content/drive/MyDrive/Colab Notebooks/sm/data/processed/num_unique_categories.txt


In [ ]:
# Также сохраним количество уникальных категорий как отдельный файл для удобства
num_cats_path = os.path.join(processed_data_dir, "num_unique_categories.txt")
with open(num_cats_path, 'w') as f:
    f.write(str(num_unique_categories))

print(f"Количество уникальных категорий сохранено в: {num_cats_path}")

Количество уникальных категорий сохранено в: /content/drive/MyDrive/Colab Notebooks/sm/data/processed/num_unique_categories.txt
